In [1]:
import numpy as np
import pandas as pd

votes = pd.read_csv("../data/processed/votes.csv")
res = pd.read_csv("../data/processed/resolutions.csv", parse_dates=["date"])

M = votes.pivot(index="ms_code", columns="undl_id", values="vote")
print(M.shape)   # countries x resolutions

(202, 5694)


In [2]:
MIN_COMMON = 20   # ignore pairs with too few shared votes

def agreement_matrix(M):
    A = M.to_numpy()
    valid = (~np.isnan(A)).astype(float)
    common = valid @ valid.T                     # votes both countries cast
    agree = np.zeros_like(common)
    for v in (1.0, 0.0, -1.0):
        onehot = (A == v).astype(float)
        agree += onehot @ onehot.T               # votes where both chose v
    with np.errstate(invalid="ignore", divide="ignore"):
        S = agree / common
    S[common < MIN_COMMON] = np.nan
    S = pd.DataFrame(S, index=M.index, columns=M.index)
    C = pd.DataFrame(common, index=M.index, columns=M.index)
    return S, C

In [3]:
def window(M, start, end):
    ids = res.loc[(res["date"] >= start) & (res["date"] <= end), "undl_id"]
    sub = M[M.columns.intersection(ids)]
    return sub.dropna(how="all")                 # drop non-members

sub = window(M, "2000-01-01", "2025-12-31")
S, C = agreement_matrix(sub)
print(sub.shape)
S.loc["USA"].dropna().sort_values(ascending=False).head(10)

(195, 2232)


ms_code
USA    1.000000
ISR    0.828491
FSM    0.681561
MHL    0.581530
PLW    0.579434
CAN    0.522052
GBR    0.507886
AUS    0.461712
FRA    0.456963
HUN    0.427088
Name: USA, dtype: float64